In [ ]:
# H2-KN01 (spec v1.2 "Held-out set 2: extraction", 2026-11-22): Docling 2.143.0 extraction, default settings (as
# P1-KN06), of the 53 held_out_2 documents (ViDoRe V3) on a Kaggle GPU. Sealed split: the run uses --final, and only the
# PDFs of dataset H2-KD01 (scripts/h2_kaggle_bundle.py), each checked against this commit's split and catalogue.
# Needs: GPU T4, Internet on (clone + model download), dataset H2-KD01 attached. No secrets. Estimated about 35-40 min.
COMMIT = ""  # the commit that holds this notebook, the --final gate of p1_extract_docling and H2 in pdfs/split.csv
assert COMMIT, "set COMMIT before running"
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git checkout -q {COMMIT} && git log --oneline -1 && git status --porcelain | head

In [ ]:
# The Docling install of ext-kn01 (EXT-KN01 ran Docling 2.143.0 on Kaggle), unchanged: same packages and pins, with
# Kaggle's own torch/torchvision.
!pip install -q docling==2.143.0 "docling-core[chunking]==2.106.0" jsonschema langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 langchain-community==0.4.2 sentence-transformers==6.2.0 transformers==5.22.1 stanza==1.15.0 google-genai==2.34.0
!pip freeze > /kaggle/working/h2-kn01_pip_freeze.txt

In [ ]:
# Fail fast, before any heavy work: the extraction script's imports, Docling 2.143.0, the GPU, the default converter
# loaded and page 1 of one dataset PDF converted (layout, table and OCR models all load).
!grep -i -E "^(docling|docling-core|docling-ibm-models|torch|torchvision|transformers)==" /kaggle/working/h2-kn01_pip_freeze.txt
import hashlib
import json
import shutil
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd

REPO, INPUT, PDF_DIR = Path("/kaggle/working/repo"), Path("/kaggle/input"), Path("/kaggle/temp/h2-pdfs")
sys.path.insert(0, str(REPO))
import scripts.p1_extract_docling
from src.parse.docling_parser import make_converter

assert version("docling") == "2.143.0", f"Docling {version('docling')}, not 2.143.0"
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "turn the GPU on"
record_path = next(INPUT.rglob("BUNDLE.json"), None)
assert record_path, "attach the dataset aman125/h2-kd01-heldout2 (BUNDLE.json not found under /kaggle/input)"
probe = record_path.parent / json.loads(record_path.read_text())["files"][0]["path"]
converter, parser_info = make_converter()
result = converter.convert(str(probe), page_range=(1, 1))
assert result.status.name == "SUCCESS" and len(result.document.pages) == 1, f"Docling probe failed: {result.status}"
print(f"imports OK | Docling {version('docling')} | converter loaded | {probe.name}: page 1 converted, "
      f"{len(result.document.export_to_text())} characters | {parser_info['settings']}")
del converter, result

In [ ]:
# Copy H2-KD01 to /kaggle/temp/h2-pdfs as <doc_id>.pdf. The bundle must hold exactly the held_out_2 documents of this
# commit's pdfs/split.csv, each with the SHA-256 in pdfs/catalog.csv; anything else stops the notebook before the run.
record, bundle = json.loads(record_path.read_text()), record_path.parent
split, catalog = pd.read_csv(REPO / "pdfs" / "split.csv"), pd.read_csv(REPO / "pdfs" / "catalog.csv").set_index("doc_id")
held_out_2 = set(split.loc[split["split"] == "held_out_2", "doc_id"])
found = {path.stem for path in bundle.rglob("*.pdf")}
assert len(held_out_2) == 53 and set(record["documents"]) == held_out_2 == found, "H2-KD01 differs from held_out_2"
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
PDF_DIR.mkdir(parents=True, exist_ok=True)
for entry in record["files"]:
    source = bundle / entry["path"]
    assert sha(source) == entry["sha256"] == catalog.loc[entry["doc_id"], "sha256"], entry["path"]
    shutil.copyfile(source, PDF_DIR / f"{entry['doc_id']}.pdf")
print(len(record["files"]), "PDFs copied and checked |", record["pages"], "pages")

In [ ]:
# Docling extraction, frozen default settings (as P1-KN06), run source H2-KN01 -> run record P1-H2-KN01-...
!cd /kaggle/working/repo && python -m scripts.p1_extract_docling --input {PDF_DIR} --output data/extracted/docling --machine kaggle --run-source H2-KN01 --final

In [ ]:
# One zip of the 53 extractions and the run record, at repo paths (unzip at the repo root on the laptop).
import zipfile

files = sorted((REPO / "data" / "extracted" / "docling").glob("*.json")) + sorted((REPO / "runs").glob("P1-H2-KN01-*.json"))
assert {path.stem for path in files if path.parent.name == "docling"} == held_out_2, "extractions missing or extra"
with zipfile.ZipFile("/kaggle/working/h2-kn01_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(REPO))
    archive.write("/kaggle/working/h2-kn01_pip_freeze.txt", "data/external/kaggle_out/H2-KN01/h2-kn01_pip_freeze.txt")
run = json.loads(max((REPO / "runs").glob("P1-H2-KN01-*.json")).read_text())
stages = pd.DataFrame(run["costs"]["stages"]).assign(corpus=lambda df: df["stage"].str.split("_").str[1])
print(run["run_id"], "|", len(files), "files zipped | wall seconds:", run["costs"]["totals"]["wall_seconds"])
print(stages.groupby("corpus")[["chars", "empty_pages", "wall_seconds"]].sum())
!ls -la /kaggle/working